Final analysis only. Select at least three accepted failures per group when available; never change thresholds to obtain examples.

In [ ]:
def failure_analysis(score_map,thresholds,output):
    ds,groups=unknown_data()
    known_names=['airplane','automobile','bird','cat','deer','dog','frog','horse','ship','truck']
    tau=thresholds['vanilla/mls']['threshold']
    records=[];chosen=[]
    for group in ['near','far']:
        a=read_cache('vanilla',group);u=score_map['vanilla'][group]['mls']
        order=np.flatnonzero(u<=tau)
        order=order[np.argsort(u[order],kind='stable')]
        # Prefer distinct unknown classes, then fill deterministically by confidence.
        picks=[];seen=set()
        for i in order:
            label=int(a['labels'][i])
            if label not in seen:picks.append(int(i));seen.add(label)
            if len(picks)==3:break
        for i in order:
            if len(picks)>=3:break
            if int(i) not in picks:picks.append(int(i))
        if len(picks)<3:print(f'Only {len(picks)} accepted {group} unknowns exist; report this honestly.')
        for i in order:
            record=dict(group=group,index=int(a['indices'][i]),unknown_class=ds.classes[int(a['labels'][i])],predicted_known_class=known_names[int(a['logits'][i].argmax())],score=float(u[i]),threshold=tau,selected_for_figure=int(i) in picks,semantic_interpretation='Review image: plausible confusion or surprising failure; explain why.')
            records.append(record)
            if int(i) in picks:chosen.append(record)
    frame=pd.DataFrame(records,columns=['group','index','unknown_class','predicted_known_class','score','threshold','selected_for_figure','semantic_interpretation'])
    frame.to_csv(output/'vanilla_mls_all_accepted_unknowns.csv',index=False)
    selected=pd.DataFrame(chosen,columns=frame.columns)
    selected.to_csv(output/'failure_examples.csv',index=False)
    if chosen:
        fig,axes=plt.subplots(2,3,figsize=(13,8),squeeze=False)
        for ax in axes.flat:ax.axis('off')
        for row,group in enumerate(['near','far']):
            for ax,record in zip(axes[row],[r for r in chosen if r['group']==group]):
                ax.imshow(ds.data[record['index']]);ax.set_title(f"{group}: {record['unknown_class']} -> {record['predicted_known_class']}\nu={record['score']:.4f}; tau={tau:.4f}",fontsize=10)
        fig.tight_layout();fig.savefig(output/'failure_examples.png',dpi=180);plt.show()
    # Keep user-written analysis safe when evaluation is re-run.
    notes=output/'failure_interpretation.md'
    if not notes.exists():
        notes.write_text('# Visual interpretation to complete after execution\n\n'+''.join(f"- {r['group']} index {r['index']}: {r['unknown_class']} -> {r['predicted_known_class']}. Plausible or surprising? Explain the visual evidence.\n" for r in chosen),encoding='utf-8')
    return selected